# Практическая работа №3: Многоресурсные системы — Ресторан с официантами и кухней

## Теоретическая справка
В реальных системах процессы часто требуют нескольких ресурсов одновременно или последовательно. Важно избегать взаимных блокировок (deadlock), запрашивая ресурсы в одинаковом порядке. В этой работе моделируется ресторан:
- Клиент сначала занимает столик
- Затем вызывает официанта для заказа
- Официант передаёт заказ на кухню
- После приготовления официант доставляет блюдо

### Старый код

In [ ]:
!pip install simpy

In [3]:
import simpy
import random

# Процесс: клиент в ресторане
def client(env, name, tables, waiters, kitchen):
    print(f'{env.now:5.1f} | {name} пришёл в ресторан')

    # 1. Занять столик
    with tables.request() as table_req:
        yield table_req
        print(f'{env.now:5.1f} | {name} сел за столик')

        # 2. Вызвать официанта для заказа
        with waiters.request() as waiter_req:
            yield waiter_req
            print(f'{env.now:5.1f} | Официант принял заказ от {name}')
            yield env.timeout(1)  # Время оформления заказа

        # 3. Приготовление на кухне (не занимаем официанта)
        with kitchen.request() as cook_req:
            yield cook_req
            cook_time = random.uniform(3, 6)
            print(f'{env.now:5.1f} | Кухня готовит заказ {name} ({cook_time:.1f} мин)')
            yield env.timeout(cook_time)

        # 4. Доставка блюда официантом
        with waiters.request() as waiter_req:
            yield waiter_req
            print(f'{env.now:5.1f} | Официант доставил блюдо {name}')
            yield env.timeout(0.5)

        # 5. Время трапезы
        yield env.timeout(random.uniform(10, 15))
        print(f'{env.now:5.1f} | {name} покинул ресторан')

def client_generator(env, tables, waiters, kitchen):
    i = 0
    while env.now < 30:
        yield env.timeout(random.expovariate(1/5))  # Клиент каждые ~5 мин
        i += 1
        env.process(client(env, f'Клиент {i}', tables, waiters, kitchen))

# Инициализация среды и ресурсов
env = simpy.Environment()
tables = simpy.Resource(env, capacity=4)    # 4 столика
waiters = simpy.Resource(env, capacity=2)   # 2 официанта
kitchen = simpy.Resource(env, capacity=3)   # 3 повара на кухне

# Запуск симуляции
env.process(client_generator(env, tables, waiters, kitchen))
env.run(until=40)

  6.4 | Клиент 1 пришёл в ресторан
  6.4 | Клиент 1 сел за столик
  6.4 | Официант принял заказ от Клиент 1
  7.4 | Кухня готовит заказ Клиент 1 (4.8 мин)
  7.7 | Клиент 2 пришёл в ресторан
  7.7 | Клиент 2 сел за столик
  7.7 | Официант принял заказ от Клиент 2
  8.7 | Кухня готовит заказ Клиент 2 (5.1 мин)
 12.2 | Официант доставил блюдо Клиент 1
 13.8 | Официант доставил блюдо Клиент 2
 19.8 | Клиент 3 пришёл в ресторан
 19.8 | Клиент 3 сел за столик
 19.8 | Официант принял заказ от Клиент 3
 20.8 | Кухня готовит заказ Клиент 3 (5.1 мин)
 24.4 | Клиент 1 покинул ресторан
 25.5 | Клиент 2 покинул ресторан
 25.9 | Официант доставил блюдо Клиент 3
 31.2 | Клиент 4 пришёл в ресторан
 31.2 | Клиент 4 сел за столик
 31.2 | Официант принял заказ от Клиент 4
 32.2 | Кухня готовит заказ Клиент 4 (4.4 мин)
 36.6 | Официант доставил блюдо Клиент 4
 40.0 | Клиент 3 покинул ресторан


### Новый код (задания 1 и 2)

In [4]:
import simpy
import random

times = []

def client(env, name, tables, waiters, kitchen):
    print(f'{env.now:5.1f} | {name} пришёл в ресторан')

    arrival_time = env.now

    # === 1. Ожидание столика (с ограничением 3 минуты) ===
    table_req = tables.request()
    result = yield table_req | env.timeout(3)

    if table_req not in result:
        print(f'{env.now:5.1f} | {name} ушёл (не дождался столика)')
        return
    else:
        print(f'{env.now:5.1f} | {name} сел за столик')

    # === 2. Ожидание официанта (для заказа) ===
    start_waiting = env.now
    with waiters.request() as waiter_req:
        yield waiter_req

        yield env.timeout(1)
        print(f'{env.now:5.1f} | Официант принял заказ от {name}')

    # === 3. Ожидание кухни ===
    with kitchen.request() as cook_req:
        yield cook_req

        cook_time = random.uniform(3, 6)
        print(f'{env.now:5.1f} | Кухня готовит заказ {name} ({cook_time:.1f} мин)')
        yield env.timeout(cook_time)

    # === 4. Официант приносит блюдо ===
    with waiters.request() as waiter_req:
        yield waiter_req
        waiting_time = env.now - start_waiting
        print(f'{env.now:5.1f} | {name} ждал официанта и кухню {waiting_time:.1f} мин (подача)')

        print(f'{env.now:5.1f} | Официант доставил блюдо {name}')
        yield env.timeout(0.5)

    # === 5. Еда ===
    yield env.timeout(random.uniform(10, 15))
    print(f'{env.now:5.1f} | {name} покинул ресторан')
    times.append(env.now - arrival_time)


def client_generator(env, tables, waiters, kitchen):
    i = 0
    while env.now < 30:
        yield env.timeout(random.expovariate(1/5))
        i += 1
        env.process(client(env, f'Клиент {i}', tables, waiters, kitchen))


# === Инициализация ===
env = simpy.Environment()
tables = simpy.Resource(env, capacity=4)
waiters = simpy.Resource(env, capacity=2)
kitchen = simpy.Resource(env, capacity=3)

# === Запуск ===
env.process(client_generator(env, tables, waiters, kitchen))
env.run(until=40)

if times:
    avg_times = sum(times) / len(times)
    print(f'Среднее время обслуживания клиента: {avg_times:.2f} мин')

  4.1 | Клиент 1 пришёл в ресторан
  4.1 | Клиент 1 сел за столик
  5.1 | Официант принял заказ от Клиент 1
  5.1 | Кухня готовит заказ Клиент 1 (3.7 мин)
  8.8 | Клиент 1 ждал официанта и кухню 4.7 мин (подача)
  8.8 | Официант доставил блюдо Клиент 1
 13.0 | Клиент 2 пришёл в ресторан
 13.0 | Клиент 2 сел за столик
 13.9 | Клиент 3 пришёл в ресторан
 13.9 | Клиент 3 сел за столик
 14.0 | Официант принял заказ от Клиент 2
 14.0 | Кухня готовит заказ Клиент 2 (4.6 мин)
 14.9 | Официант принял заказ от Клиент 3
 14.9 | Кухня готовит заказ Клиент 3 (4.9 мин)
 16.8 | Клиент 4 пришёл в ресторан
 16.8 | Клиент 4 сел за столик
 17.8 | Официант принял заказ от Клиент 4
 17.8 | Кухня готовит заказ Клиент 4 (5.9 мин)
 18.6 | Клиент 2 ждал официанта и кухню 5.6 мин (подача)
 18.6 | Официант доставил блюдо Клиент 2
 19.8 | Клиент 3 ждал официанта и кухню 5.9 мин (подача)
 19.8 | Официант доставил блюдо Клиент 3
 21.0 | Клиент 5 пришёл в ресторан
 22.3 | Клиент 1 покинул ресторан
 23.1 | Клиент 6 

### Новый код (задания 3 и 4)

In [5]:
import simpy
import random

times = []

def client(env, name, tables, waiters, kitchen):
    print(f'{env.now:5.1f} | {name} пришёл в ресторан')

    arrival_time = env.now

    # === 1. Ожидание столика (с ограничением 3 минуты) ===
    table_req = tables.request()
    result = yield table_req | env.timeout(3)

    if table_req not in result:
        print(f'{env.now:5.1f} | {name} ушёл (не дождался столика)')
        return
    else:
        print(f'{env.now:5.1f} | {name} сел за столик')

    # === 2–4. Официант остаётся с клиентом ===
    start_waiting = env.now
    with waiters.request() as waiter_req:
        yield waiter_req

        # заказ
        yield env.timeout(1)
        print(f'{env.now:5.1f} | Официант принял заказ от {name}')

        # кухня (официант ЖДЁТ вместе с клиентом)
        with kitchen.request() as cook_req:
            yield cook_req

            cook_time = random.uniform(3, 6)
            print(f'{env.now:5.1f} | Кухня готовит заказ {name} ({cook_time:.1f} мин)')
            yield env.timeout(cook_time)

        # подача (официант уже здесь)
        waiting_time = env.now - start_waiting
        print(f'{env.now:5.1f} | {name} ждал официанта и кухню {waiting_time:.1f} мин (подача)')

        print(f'{env.now:5.1f} | Официант доставил блюдо {name}')
        yield env.timeout(0.5)

    # === 5. Еда ===
    yield env.timeout(random.uniform(10, 15))
    print(f'{env.now:5.1f} | {name} покинул ресторан')
    times.append(env.now - arrival_time)


def client_generator(env, tables, waiters, kitchen):
    i = 0
    while env.now < 30:
        yield env.timeout(random.expovariate(1/5))
        i += 1
        env.process(client(env, f'Клиент {i}', tables, waiters, kitchen))


# === Инициализация ===
env = simpy.Environment()
tables = simpy.Resource(env, capacity=4)
waiters = simpy.Resource(env, capacity=2)
kitchen = simpy.Resource(env, capacity=3)

# === Запуск ===
env.process(client_generator(env, tables, waiters, kitchen))
env.run(until=40)

if times:
    avg_times = sum(times) / len(times)
    print(f'Среднее время обслуживания клиента: {avg_times:.2f} мин')

  0.7 | Клиент 1 пришёл в ресторан
  0.7 | Клиент 1 сел за столик
  1.7 | Официант принял заказ от Клиент 1
  1.7 | Кухня готовит заказ Клиент 1 (4.6 мин)
  2.6 | Клиент 2 пришёл в ресторан
  2.6 | Клиент 2 сел за столик
  3.6 | Клиент 3 пришёл в ресторан
  3.6 | Клиент 3 сел за столик
  3.6 | Официант принял заказ от Клиент 2
  3.6 | Кухня готовит заказ Клиент 2 (5.7 мин)
  6.3 | Клиент 1 ждал официанта и кухню 5.6 мин (подача)
  6.3 | Официант доставил блюдо Клиент 1
  6.3 | Клиент 4 пришёл в ресторан
  6.3 | Клиент 4 сел за столик
  7.8 | Официант принял заказ от Клиент 3
  7.8 | Кухня готовит заказ Клиент 3 (3.4 мин)
  8.2 | Клиент 5 пришёл в ресторан
  9.3 | Клиент 2 ждал официанта и кухню 6.7 мин (подача)
  9.3 | Официант доставил блюдо Клиент 2
 10.8 | Официант принял заказ от Клиент 4
 10.8 | Кухня готовит заказ Клиент 4 (5.0 мин)
 11.2 | Клиент 3 ждал официанта и кухню 7.6 мин (подача)
 11.2 | Официант доставил блюдо Клиент 3
 11.2 | Клиент 6 пришёл в ресторан
 11.2 | Клиент 5

### Задание 5

In [11]:
import simpy
import random

times = []

def client(env, name, tables, waiters, kitchen):
    # print(f'{env.now:5.1f} | {name} пришёл в ресторан')

    arrival_time = env.now

    # === 1. Ожидание столика (с ограничением 3 минуты) ===
    table_req = tables.request()
    result = yield table_req | env.timeout(3)

    # if table_req not in result:
    #     print(f'{env.now:5.1f} | {name} ушёл (не дождался столика)')
    #     return
    # else:
    #     print(f'{env.now:5.1f} | {name} сел за столик')

    # === 2–4. Официант остаётся с клиентом ===
    start_waiting = env.now
    with waiters.request() as waiter_req:
        yield waiter_req

        # заказ
        yield env.timeout(1)
        # print(f'{env.now:5.1f} | Официант принял заказ от {name}')

        # кухня (официант ЖДЁТ вместе с клиентом)
        with kitchen.request() as cook_req:
            yield cook_req

            cook_time = random.uniform(3, 6)
            # print(f'{env.now:5.1f} | Кухня готовит заказ {name} ({cook_time:.1f} мин)')
            yield env.timeout(cook_time)

        # подача (официант уже здесь)
        waiting_time = env.now - start_waiting
        # print(f'{env.now:5.1f} | {name} ждал официанта и кухню {waiting_time:.1f} мин (подача)')

        # print(f'{env.now:5.1f} | Официант доставил блюдо {name}')
        yield env.timeout(0.5)

    # === 5. Еда ===
    yield env.timeout(random.uniform(10, 15))
    # print(f'{env.now:5.1f} | {name} покинул ресторан')
    times.append(env.now - arrival_time)


def client_generator(env, tables, waiters, kitchen):
    i = 0
    while env.now < 30:
        yield env.timeout(random.expovariate(1/5))
        i += 1
        env.process(client(env, f'Клиент {i}', tables, waiters, kitchen))


# функция для сравнения разных конфигураций поваров и официантов
def run_sim(waiters_count, kitchen_count, runs=5):
    results = []

    for _ in range(runs):
        global times
        times = []
        # === Инициализация ===
        env = simpy.Environment()
        tables = simpy.Resource(env, capacity=4)
        waiters = simpy.Resource(env, capacity=waiters_count)
        kitchen = simpy.Resource(env, capacity=kitchen_count)

        # === Запуск ===
        env.process(client_generator(env, tables, waiters, kitchen))
        env.run(until=40)

        if times:
            results.append(sum(times) / len(times))

    return sum(results) / len(results) if results else 0

configs = [
    (2, 2),
    (2, 3),
    (3, 2),
    (3, 3),
    (4, 3),
    (3, 4),
]

print("\nСравнение конфигураций:")
for w, k in configs:
    avg = run_sim(w, k)
    print(f'Официанты: {w}, Повара: {k} → {avg:.2f} мин')




Сравнение конфигураций:
Официанты: 2, Повара: 2 → 19.51 мин
Официанты: 2, Повара: 3 → 19.47 мин
Официанты: 3, Повара: 2 → 18.67 мин
Официанты: 3, Повара: 3 → 18.47 мин
Официанты: 4, Повара: 3 → 18.03 мин
Официанты: 3, Повара: 4 → 19.24 мин


## Задание для самостоятельной работы
1. Добавьте учёт времени ожидания официанта и кухни отдельно для каждого клиента
2. Введите ограничение: клиент уходит, если не дождался столика за 3 минуты (используйте `yield env.timeout() | req`)
3. Измените логику: официант остаётся у столика во время приготовления (занимает ресурс официанта дольше)
4. Сравните среднее время пребывания клиентов в двух сценариях (с п.3 и без)
5. Предложите оптимальное соотношение официантов и поваров для минимизации времени ожидания

### Ответы на вопросы

**1. Что такое deadlock и как его избежать?**


Взаимная блокировка — это ситуация, когда процессы ждут друг друга и никто не может продолжить работу.
- Избежать можно, если всегда запрашивать ресурсы в одном и том же порядке или не держать один ресурс, ожидая другой.



**2. Почему важно освобождать ресурсы в обратном порядке?**

Это снижает риск блокировок и конфликтов.
- Сначала освобождается последний занятый ресурс, и другие процессы быстрее получают доступ, не создавая “цепочек ожидания”.


**3. Как моделировать уход клиента при долгом ожидании?**

`yield request | env.timeout(T)`

Если за время T ресурс не получен — клиент уходит.

**4. Почему официант не ждёт у кухни в реальности?**

Потому что это неэффективно: он простаивает и не обслуживает других клиентов.
- Разделение ресурсов увеличивает пропускную способность системы.

**5. Какие метрики важны?**



*   среднее время обслуживания клиента
*   время ожидания
*   загрузка ресурсов (официанты, кухня)
*   количество ушедших клиентов

Эти показатели помогают понять, где “узкие места” и как улучшить работу ресторана.

## Контрольные вопросы
1. Что такое взаимная блокировка (deadlock) и как её избежать при запросе нескольких ресурсов?
2. В чём преимущество последовательного освобождения ресурсов в обратном порядке запроса?
3. Как моделировать уход клиента при превышении времени ожидания?
4. Почему в реальных системах часто используется разделение ресурсов (официант не ждёт у кухни)?
5. Какие метрики эффективности важны для ресторанного бизнеса при анализе через симуляцию?